### RAG 性能评估，指标：Answer_relevancy（答案相关性）、context_precision（上下文精度）、context_recall（上下文召回率）、faithfulness（忠实度）

### 1 创建数据集

#### 1.1 读取测试数据集

In [1]:
import json
import os
import sys
from pprint import pprint

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from utils.path_tool import get_abs_path


data_path = get_abs_path("eval/data.json")

data_dict = json.load(open(data_path, "r", encoding="utf-8"))
data_dict["question"] = data_dict["question"][:5]
data_dict["ground_truth"] = data_dict["ground_truth"][:5]

In [12]:
data_dict

{'question': ['首次使用扫地机器人前，正确的准备步骤是什么？', '机器人总是找不到充电座，充电座周围需要预留多大空间？'],
 'answer': [],
 'contexts': [],
 'ground_truth': ['拆除机身所有包装配件，先把机器人充满电，再在空旷环境下启动建图；建图完成后再设置清扫区域和禁区。',
  '应清理充电座周围障碍物，保证充电座前方3米、左右各1米无遮挡，再把机器人放回充电座附近重启回充。']}

#### 1.2 检索文本、生成答案

In [2]:
from rag.rag_service import RAGSummarizeService
from tqdm import tqdm


rag = RAGSummarizeService()

for query in tqdm(data_dict["question"]):
    answer, context = rag.rag_summarize_with_context(query)
    data_dict["answer"].append(answer)
    data_dict["contexts"].append([context])

c:\Users\player\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


#### 1.3 转换为 Dataset 对象

In [3]:
from datasets import Dataset

dataset = Dataset.from_dict(data_dict)
print(f'数据集共 {len(dataset)} 条记录')
for i in range(min(5, len(dataset))):
    print(f'----------------第{i}条记录:-------------------')
    for key, value in dataset[i].items():
        print(f'{key}: {value}')
   


数据集共 5 条记录
----------------第0条记录:-------------------
question: 首次使用扫地机器人前，正确的准备步骤是什么？
answer: 拆除机身所有包装配件，充满电后，在空旷环境下启动建图，完成后再设置清扫区域和禁区。
contexts: ["【参考资料1】：# 扫地机器人常见问题及解答（100条）\n### 基础使用类\n1. **首次使用扫地机器人需要做什么？**\n- 拆除机身所有包装配件，充满电后，在空旷环境下启动建图，完成后再设置清扫区域和禁区。\n2. **开机后机器人不移动怎么办？**\n- 检查机身是否被防撞条、电源线缠绕，确认底部万向轮、驱动轮无卡顿，重启设备重试。\n3. **机器人找不到充电座怎么处理？** | 参考元数据：{'source': 'E:\\\\workspace\\\\Agent项目实战\\\\data\\\\扫地机器人100问2.txt'}\n\n【参考资料2】：# 扫地机器人+扫拖一体机器人 维护保养200条（纯保养维度，分通用基础/扫地专属/扫拖一体拖地专属/耗材专项/环境适配/长期存放/故障预防，覆盖全机型日常维护）\n## 通用基础维护（50条）\n1. 每日使用后，用干软布擦拭机器人机身外壳，去除灰尘、水渍，避免污渍附着硬化。\n2. 每次清扫完成，及时清理机身防撞条缝隙的毛发、线头，防止卡扣卡顿影响回弹。 | 参考元数据：{'source': 'E:\\\\workspace\\\\Agent项目实战\\\\data\\\\维护保养.txt'}\n\n【参考资料3】：- 清理床底、沙发底的缠绕物，确保机器人可顺利进入，在APP中对该区域设置“深度拖扫”，增大吸力和拖地次数。 | 参考元数据：{'source': 'E:\\\\workspace\\\\Agent项目实战\\\\data\\\\扫拖一体机器人100问.txt'}\n\n"]
ground_truth: 拆除机身所有包装配件，先把机器人充满电，再在空旷环境下启动建图；建图完成后再设置清扫区域和禁区。
----------------第1条记录:-------------------
question: 机器人总是找不到充电座，充电座周围需要预留多大空间？
answer: 充电座前方需预留3米、左右各1

### 2 创建 LLM 实例（充当裁判）和嵌入模型

#### 2.1 创建 LLM 实例（方式一：通过 ragas 的 llm_factory）

In [4]:
from openai import OpenAI
from ragas.llms import llm_factory

client = OpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
llm = llm_factory(
    model="qwen-plus",
    provider="openai",
    client=client
)

# 测试模型是否正常工作
from pydantic import BaseModel

class UserInfo(BaseModel):
    name: str
    age: int
    city: str

prompt = """我叫张三，今年25岁，来自杭州。"""

result: UserInfo = llm.generate(prompt, response_model=UserInfo)
print(f'type(result): {type(result)}')
print(result)

type(result): <class '__main__.UserInfo'>
name='张三' age=25 city='杭州'


#### 2.2 创建 LLM 实例（方式二：通过 langchain_community.chat_models）
 

In [4]:
from langchain_community.chat_models.tongyi import ChatTongyi

llm = ChatTongyi(model="qwen-max")

# 测试模型是否正常工作
result = llm.invoke("你好")
print(result)

content='你好！有什么可以帮助你的吗？' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '23ec38dc-9996-9662-8bab-6011ffeca24f', 'token_usage': {'input_tokens': 9, 'output_tokens': 7, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 16}} id='lc_run--019d581d-b1dc-7de3-b5db-8c8d29d317a5-0' tool_calls=[] invalid_tool_calls=[]


#### 2.3 创建嵌入模型

In [5]:
from langchain_community.embeddings import DashScopeEmbeddings

embedding = DashScopeEmbeddings()

### 3 内容解析器，提取 token usage

In [6]:
from ragas.cost import TokenUsage
from langchain_core.outputs import LLMResult

total_input_tokens = 0
total_output_tokens = 0


# 查看模型输出结构
def debug_token_usage_parser(result: LLMResult) -> TokenUsage:
    # print("llm_output:", result.model_dump())  # 先看结构
    try:
        token_data = result.model_dump()["generations"][0][0]["generation_info"]["token_usage"]
        # print("token_data:", token_data)
    except KeyError:
        token_data = {}
    # 更新全局 token 统计
    global total_input_tokens, total_output_tokens
    total_input_tokens += token_data.get("input_tokens", 0)
    total_output_tokens += token_data.get("output_tokens", 0)
    return TokenUsage(
        input_tokens=token_data.get("input_tokens", 0),
        output_tokens=token_data.get("output_tokens", 0),
    )


def openai_token_usage_parser(result: LLMResult) -> TokenUsage:
    usage = result.llm_output.get("token_usage", {})
    return TokenUsage(
        input_tokens=usage.get("prompt_tokens", 0),
        output_tokens=usage.get("completion_tokens", 0),
    )

### 4 开始评估

In [7]:
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, faithfulness, context_recall

metrics = [answer_relevancy, context_precision, faithfulness, context_recall]
# 尝试将 Prompt 适配为中文，极大缓解大模型中英夹杂时的理解偏差
for metric in metrics:
    if hasattr(metric, 'adapt'):
        print(f"Adapting {metric.__name__} to Chinese...")
        metric.adapt("chinese", llm=llm)

total_input_tokens = 0
total_output_tokens = 0

result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=llm,
    embeddings=embedding,
    experiment_name="rag_eval",
    # batch_size=8,
    raise_exceptions=True, # 强烈建议开启，防止大模型JSON输出错误被静默记为0分
    token_usage_parser=debug_token_usage_parser
)
print(result)
print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")


C:\Users\player\AppData\Local\Temp\ipykernel_27808\1654083952.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, context_precision, faithfulness, context_recall
C:\Users\player\AppData\Local\Temp\ipykernel_27808\1654083952.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import answer_relevancy, context_precision, faithfulness, context_recall
C:\Users\player\AppData\Local\Temp\ipykernel_27808\1654083952.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Examp

{'answer_relevancy': 0.5928, 'context_precision': 1.0000, 'faithfulness': 1.0000, 'context_recall': 1.0000}
Total input tokens: 29045
Total output tokens: 2963


### 5 保存结果

In [8]:
# print(type(result))
import datetime
result_path = get_abs_path(f"eval/result/rag_eval_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

# 保存为 CSV（最方便，后续可以用 Excel / pandas 分析）
df = result.to_pandas()          # 转换为 DataFrame，每一行对应一个样本的详细得分
df.to_csv(result_path, index=True)

print(f"结果已保存到 {result_path}")


结果已保存到 e:\workspace\Agent项目实战\eval/result/rag_eval_20260404_192810.csv
